# Демо 1. Класичне донавчання: зміна голови в моделі BERT

У цьому ноутбуці донавчаємо модель **BERT** для задачі **визначення парафраз**: модель отримує два речення й визначає, чи передають вони той самий зміст.

План:
1. Завантажити датасет MRPC і токенізувати пари речень.
2. Завантажити BERT із **новою головою**-класифікатором.
3. **Повне донавчання** з `Trainer`: тренуються всі ваги (кістяк і голова).
4. Перевірити модель на власних прикладах.

Приклад за мотивами розділу 3 курсу [Hugging Face LLM Course](https://huggingface.co/learn/llm-course/chapter3/1).
Для запуску в Colab оберіть середовище з GPU: *Runtime → Change runtime type → T4 GPU*.

**Некрасов Артур Дмитрович**

У GPU-профілі використовуються повні набори MRPC. У CPU-профілі використано фіксовані підвибірки. Час кожної клітинки виводиться як `Wall time`.

In [ ]:
%%time
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("pip", 'install -q "transformers==5.20.2" "datasets==5.1.0" "accelerate==1.15.0"')

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.
CPU times: user 3.29 s, sys: 793 ms, total: 4.08 s
Wall time: 40.3 s


In [ ]:
%%time
import numpy as np
import torch
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

CPU times: user 19 s, sys: 1.88 s, total: 20.9 s
Wall time: 35.1 s


In [ ]:
%%time
set_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
CPU_PROFILE = device == "cpu"
if CPU_PROFILE:
    torch.set_num_threads(4)
print("Пристрій:", device)
print("Профіль:", "CPU, train=256 і validation=128" if CPU_PROFILE else "GPU, повні train і validation")
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Пристрій: cuda
Профіль: GPU, повні train і validation
GPU: Tesla T4
CPU times: user 10.4 ms, sys: 111 ms, total: 122 ms
Wall time: 229 ms


## 1. Дані: MRPC

**MRPC** (Microsoft Research Paraphrase Corpus) — пари речень з новинних статей.
Мітка `1` означає, що речення є парафразами, мітка `0` — що ні.

In [ ]:
%%time
raw_datasets = load_dataset("nyu-mll/glue", "mrpc")
raw_datasets

README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

mrpc/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  649kB            

mrpc/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

mrpc/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 75.7kB            

mrpc/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

mrpc/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  308kB            

... (2 outputs left out)

Generating validation split:   0%|          | 0/408 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1725 [00:00<?, ? examples/s]

CPU times: user 1.32 s, sys: 134 ms, total: 1.45 s
Wall time: 5.98 s


DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [ ]:
%%time
# Подивимося на один приклад
print("Приклад нееквівалентних речень:")
example = raw_datasets["train"][3]
print("Речення 1:", example["sentence1"])
print("Речення 2:", example["sentence2"])
print("Мітка:   ", example["label"], "→", raw_datasets["train"].features["label"].names[example["label"]])

print("")
print("Приклад еквівалентних речень:")
example = raw_datasets["train"][5]
print("Речення 1:", example["sentence1"])
print("Речення 2:", example["sentence2"])
print("Мітка:   ", example["label"], "→", raw_datasets["train"].features["label"].names[example["label"]])

Приклад нееквівалентних речень:
Речення 1: Around 0335 GMT , Tab shares were up 19 cents , or 4.4 % , at A $ 4.56 , having earlier set a record high of A $ 4.57 .
Речення 2: Tab shares jumped 20 cents , or 4.6 % , to set a record closing high at A $ 4.57 .
Мітка:    0 → not_equivalent

Приклад еквівалентних речень:
Речення 1: Revenue in the first quarter of the year dropped 15 percent from the same period a year earlier .
Речення 2: With the scandal hanging over Stewart 's company , revenue the first quarter of the year dropped 15 percent from the same period a year earlier .
Мітка:    1 → equivalent
CPU times: user 1.34 ms, sys: 0 ns, total: 1.34 ms
Wall time: 1.35 ms


### Знайомимось з BERT BASE UNCASED

Для цього прочитайте інформацію на [картці моделі](https://huggingface.co/google-bert/bert-base-uncased)

In [ ]:
%%time
from transformers import pipeline

unmasker = pipeline('fill-mask', model='google-bert/bert-base-uncased', device=0 if device == "cuda" else -1)
mask_man = unmasker("The man worked as a [MASK].")
display(mask_man)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: google-bert/bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

[{'score': 0.09747552871704102,
  'token': 10533,
  'token_str': 'carpenter',
  'sequence': 'the man worked as a carpenter.'},
 {'score': 0.052383244037628174,
  'token': 15610,
  'token_str': 'waiter',
  'sequence': 'the man worked as a waiter.'},
 {'score': 0.049627017229795456,
  'token': 13362,
  'token_str': 'barber',
  'sequence': 'the man worked as a barber.'},
 {'score': 0.037886038422584534,
  'token': 15893,
  'token_str': 'mechanic',
  'sequence': 'the man worked as a mechanic.'},
 {'score': 0.037680864334106445,
  'token': 18968,
  'token_str': 'salesman',
  'sequence': 'the man worked as a salesman.'}]

CPU times: user 2.12 s, sys: 1.35 s, total: 3.47 s
Wall time: 8.38 s


In [ ]:
%%time
mask_woman = unmasker("The woman worked as a [MASK].")
display(mask_woman)
del unmasker

[{'score': 0.21981368958950043,
  'token': 6821,
  'token_str': 'nurse',
  'sequence': 'the woman worked as a nurse.'},
 {'score': 0.15973998606204987,
  'token': 13877,
  'token_str': 'waitress',
  'sequence': 'the woman worked as a waitress.'},
 {'score': 0.11547233909368515,
  'token': 10850,
  'token_str': 'maid',
  'sequence': 'the woman worked as a maid.'},
 {'score': 0.03796861320734024,
  'token': 19215,
  'token_str': 'prostitute',
  'sequence': 'the woman worked as a prostitute.'},
 {'score': 0.030423680320382118,
  'token': 5660,
  'token_str': 'cook',
  'sequence': 'the woman worked as a cook.'}]

CPU times: user 29.2 ms, sys: 1.2 ms, total: 30.4 ms
Wall time: 50.2 ms


### Токенізація пари речень

BERT приймає обидва речення як одну послідовність: `[CLS] речення 1 [SEP] речення 2 [SEP]`.
Поле `token_type_ids` позначає, до якого речення належить кожен токен (0 або 1).

Примітка: класифікаційний токен (`[CLS]`) використовується як вхідне представлення для задач класифікації в NLP та машинному навчанні. Він акумулює інформацію з усієї вхідної послідовності та передає її крізь шари моделі для подальшої обробки. Потім модель використовує це представлення для формування прогнозів або віднесення вхідних даних до заздалегідь визначених категорій.

In [ ]:
%%time
checkpoint = "google-bert/bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)
encoded = tokenizer(example["sentence1"], example["sentence2"])

print("")
print(tokenizer.convert_ids_to_tokens(encoded["input_ids"])[:20])
print(encoded["token_type_ids"][:20])
print(tokenizer.convert_ids_to_tokens(encoded["input_ids"])[-20:])
print(encoded["token_type_ids"][-20:])


['[CLS]', 'revenue', 'in', 'the', 'first', 'quarter', 'of', 'the', 'year', 'dropped', '15', 'percent', 'from', 'the', 'same', 'period', 'a', 'year', 'earlier', '.']
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[',', 'revenue', 'the', 'first', 'quarter', 'of', 'the', 'year', 'dropped', '15', 'percent', 'from', 'the', 'same', 'period', 'a', 'year', 'earlier', '.', '[SEP]']
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
CPU times: user 113 ms, sys: 4.75 ms, total: 118 ms
Wall time: 394 ms


In [ ]:
%%time
def tokenize_function(batch):
    return tokenizer(batch["sentence1"], batch["sentence2"], truncation=True, max_length=128)

experiment_data = raw_datasets
if CPU_PROFILE:
    from datasets import DatasetDict
    experiment_data = DatasetDict({
        "train": raw_datasets["train"].shuffle(seed=42).select(range(256)),
        "validation": raw_datasets["validation"].shuffle(seed=42).select(range(128)),
    })
tokenized_datasets = experiment_data.map(tokenize_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
display(tokenized_datasets)

Map:   0%|          | 0/3668 [00:00<?, ? examples/s]

Map:   0%|          | 0/408 [00:00<?, ? examples/s]

Map:   0%|          | 0/1725 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})

CPU times: user 2.65 s, sys: 57.9 ms, total: 2.71 s
Wall time: 3 s


## 2. Модель: кістяк + нова голова

`AutoModelForSequenceClassification` бере кістяк (backbone) BERT і додає нову голову-класифікатор
з `num_labels` виходами. Зверніть увагу на звіт під час завантаження:
- ваги `cls.*` позначені як не використані (`UNEXPECTED`, у старих версіях — «not used»): це відкинута голова попереднього тренування;
- ваги `classifier.*` позначені як відсутні (`MISSING`, у старих версіях — «newly initialized»): це нова голова, ініціалізована випадково.

In [ ]:
%%time
set_seed(42)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

print(model.classifier)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Linear(in_features=768, out_features=2, bias=True)
CPU times: user 134 ms, sys: 1.44 ms, total: 135 ms
Wall time: 244 ms


In [ ]:
%%time
def count_params(module, trainable_only=False):
    return sum(p.numel() for p in module.parameters() if p.requires_grad or not trainable_only)


print(f"Кістяк (model.bert):   {count_params(model.bert):>12,} параметрів")
print(f"Голова (model.classifier): {count_params(model.classifier):>8,} параметрів")

Кістяк (model.bert):    109,482,240 параметрів
Голова (model.classifier):    1,538 параметрів
CPU times: user 1.94 ms, sys: 0 ns, total: 1.94 ms
Wall time: 2.59 ms


## 3. Повне донавчання з `Trainer`

`Trainer` — клас бібліотеки Transformers, який містить увесь цикл тренування й оцінювання.
Метрики рахуємо самі у функції `compute_metrics`.

Оновлюються **всі** ваги: і кістяк, і нова голова.
Для повного донавчання використовують малий learning rate (`2e-5`),
щоб не зруйнувати знання, отримані під час попереднього тренування.

In [ ]:
%%time
import gc
import json
import time
from pathlib import Path

import pandas as pd

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions, zero_division=0),
    }

def make_trainer(classifier, epochs):
    args = TrainingArguments(
        output_dir=f"bert-mrpc-{epochs}",
        num_train_epochs=epochs,
        learning_rate=2e-5,
        per_device_train_batch_size=8 if CPU_PROFILE else 16,
        per_device_eval_batch_size=16 if CPU_PROFILE else 64,
        eval_strategy="epoch",
        save_strategy="no",
        logging_strategy="epoch",
        fp16=device == "cuda",
        dataloader_pin_memory=device == "cuda",
        report_to="none",
        disable_tqdm=True,
        seed=42,
        data_seed=42,
    )
    return Trainer(
        model=classifier,
        args=args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["validation"],
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

trainer = make_trainer(model, 3)

CPU times: user 141 ms, sys: 52.6 ms, total: 194 ms
Wall time: 232 ms


In [ ]:
%%time
print(f"Параметрів, що тренуються: {count_params(model, trainable_only=True):,}")
started = time.perf_counter()
trainer.train()
training_3_s = time.perf_counter() - started
history_3 = list(trainer.state.log_history)
evaluation_3 = trainer.evaluate()
print(f"3 епохи: {training_3_s:.2f} с")
display(pd.DataFrame([evaluation_3]).round(4))

Параметрів, що тренуються: 109,483,778
{'loss': '0.5317', 'grad_norm': '15.38', 'learning_rate': '1.336e-05', 'epoch': '1'}
{'eval_loss': '0.3815', 'eval_accuracy': '0.8382', 'eval_f1': '0.887', 'eval_runtime': '0.459', 'eval_samples_per_second': '888.8', 'eval_steps_per_second': '15.25', 'epoch': '1'}
{'loss': '0.3101', 'grad_norm': '4.81', 'learning_rate': '6.696e-06', 'epoch': '2'}
{'eval_loss': '0.3576', 'eval_accuracy': '0.848', 'eval_f1': '0.8935', 'eval_runtime': '0.4207', 'eval_samples_per_second': '969.7', 'eval_steps_per_second': '16.64', 'epoch': '2'}
{'loss': '0.1626', 'grad_norm': '0.9043', 'learning_rate': '2.899e-08', 'epoch': '3'}
{'eval_loss': '0.4165', 'eval_accuracy': '0.8603', 'eval_f1': '0.9016', 'eval_runtime': '0.462', 'eval_samples_per_second': '883.2', 'eval_steps_per_second': '15.15', 'epoch': '3'}
{'train_runtime': '85.11', 'train_samples_per_second': '129.3', 'train_steps_per_second': '8.107', 'train_loss': '0.3348', 'epoch': '3'}
{'eval_loss': '0.4165', 'ev

,eval_loss,eval_accuracy,eval_f1,eval_runtime,eval_samples_per_second,eval_steps_per_second,epoch
0,0.4165,0.8603,0.9016,0.4766,856.095,14.688,3.0


CPU times: user 57.5 s, sys: 528 ms, total: 58 s
Wall time: 1min 26s


## 4. Перевірка на власних прикладах

In [ ]:
%%time
def predict(model, sentence1, sentence2):
    model.eval()
    inputs = tokenizer(sentence1, sentence2, return_tensors="pt").to(model.device)
    with torch.no_grad():
        probs = model(**inputs).logits.softmax(dim=-1)[0]
    return f"парафраз: {probs[1]:.1%}"


pairs = [
    ("The company reported a profit of $5 million.", "The firm earned five million dollars in profit."),
    ("The company reported a profit of $5 million.", "The weather was sunny all week."),
    ("She bought a new car yesterday.", "Yesterday she purchased a new vehicle."),
]
for s1, s2 in pairs:
    print(f"{s1}\n{s2}\n→ {predict(model, s1, s2)}\n")

The company reported a profit of $5 million.
The firm earned five million dollars in profit.
→ парафраз: 86.0%

The company reported a profit of $5 million.
The weather was sunny all week.
→ парафраз: 2.6%

She bought a new car yesterday.
Yesterday she purchased a new vehicle.
→ парафраз: 99.0%

CPU times: user 76.2 ms, sys: 7.72 ms, total: 83.9 ms
Wall time: 99.8 ms


In [ ]:
%%time
extra_pairs = [
    ("The library opens at nine on weekdays.", "On workdays the library starts serving visitors at 9 a.m.", 1),
    ("The board postponed the meeting until Friday.", "The meeting was put off by the board until Friday.", 1),
    ("The clinic vaccinated 120 children.", "The clinic vaccinated 210 children.", 0),
    ("The police officer called the journalist.", "The journalist called the police officer.", 0),
]

extra_results = []
for left, right, expected in extra_pairs:
    batch = tokenizer(left, right, return_tensors="pt", truncation=True, max_length=128).to(model.device)
    with torch.inference_mode():
        probability = float(model(**batch).logits.softmax(-1)[0, 1])
    extra_results.append({"Речення 1": left, "Речення 2": right, "Еталон": expected,
        "Передбачення": int(probability >= 0.5), "P(парафраз)": round(probability, 4)})
display(pd.DataFrame(extra_results))

,Речення 1,Речення 2,Еталон,Передбачення,P(парафраз)
0,The library opens at nine on weekdays.,On workdays the library starts serving visitor...,1,1,0.8590
1,The board postponed the meeting until Friday.,The meeting was put off by the board until Fri...,1,1,0.9777
2,The clinic vaccinated 120 children.,The clinic vaccinated 210 children.,0,1,0.9846
3,The police officer called the journalist.,The journalist called the police officer.,0,1,0.9878


CPU times: user 82.2 ms, sys: 642 µs, total: 82.9 ms
Wall time: 92.9 ms


## 0.1. Відповіді на питання 1–10

### 1. Час виконання клітинок

На Tesla T4 сума `Wall time` для 16 клітинок становила близько 4 хв 51 с. Установлення пакетів тривало 40,3 с, імпорт бібліотек — 35,1 с, завантаження MRPC — 5,98 с. Клітинки з тренуванням на 3 і 5 епох виконувалися 1 хв 26 с і 1 хв 51 с відповідно. Використано 3668 тренувальних і 408 валідаційних пар.

Під час установлення `pip` повідомив про конфлікт версій `pyarrow` і `cudf-cu13`. Імпорт бібліотек, виявлення GPU і тренування завершилися успішно.

### 2. Токен `[CLS]`

`[CLS]` стоїть на початку вхідної послідовності BERT. У кожному шарі self-attention його представлення оновлюється з урахуванням інших токенів. Класифікатор бере фінальний вектор цього токена, оброблений pooler, і обчислює logits класів. Під час донавчання градієнт від мітки пари змінює голову й кістяк, тож представлення `[CLS]` стає корисним саме для рішення про парафраз.

### 3. Пряме застосування `bert-base-uncased`

Для базової моделі з MLM-головою можна замаскувати слово й передбачити можливі заповнення `[MASK]` за двостороннім контекстом. У демо це робить `pipeline('fill-mask')`. Задача MRPC потребує окремої голови класифікації та розмічених пар. [Картка BERT](https://huggingface.co/google-bert/bert-base-uncased)

### 4. Сексистські асоціації

Так, особливо в системах, де модель впливає на відбір людей або рішення про доступ. Приклад із заміною `man` на `woman` показує, що прогноз професій може змінитися за незмінного решти контексту. У збереженому виводі для `man` перші три варіанти — `carpenter`, `waiter`, `barber`. Для `woman` — `nurse`, `waitress`, `maid`. Найімовірніші слова мають імовірності 9,75% і 21,98% відповідно. Для вимірювання потрібен набір таких пар, розподіли відповідей і ручна перевірка. Можливі дії — збалансувати приклади, донавчити на контрприкладах, перевірити показники на окремих групах і передбачити людський перегляд рішень. Два промпти демонструють асоціацію, а її поширеність треба оцінити на більшій вибірці.

### 5. Числа 0 і 1

Це `token_type_ids`, тобто ознаки сегментів. У форматі `[CLS] A [SEP] B [SEP]` токени першого сегмента мають 0, другого — 1. Відповідні ембединги додаються до токенних і позиційних ембедингів. Вони дають моделі ознаку, до якого речення належить кожен токен. Мітки еквівалентності пари зберігаються окремо в `label`.

### 6. Розміри 768 і 2

768 — розмір прихованого представлення BERT-base. 2 — кількість logits для класів «нееквівалентні» та «еквівалентні». Лінійний шар має матрицю 2 × 768 і два зміщення, разом 1538 параметрів.

### 7. Кістяк і голова

Великий кістяк уже містить мовні представлення, тож маленька нова голова може адаптувати їх до двох класів. Для невеликого MRPC це корисне перенесення знань. Повне оновлення кістяка збільшує час і пам’ять, тому кількість епох і learning rate оцінюються за валідацією. Замороження частини кістяка було б способом зменшити витрати.

### 8. Accuracy і F1

Accuracy — частка пар, яким присвоєно правильну мітку. F1 у цьому коді обчислюється для позитивного класу `1` і дорівнює гармонічному середньому precision та recall. F1 показує, як модель знаходить еквівалентні пари з урахуванням хибних спрацювань. Метрики обчислюються на тому самому валідаційному наборі MRPC для обох запусків.

### 9. Збільшення кількості епох

На Tesla T4 порівняв два запуски з однаковими наборами даних, початковими вагами та seed.

| Епохи | Час тренування, с | Accuracy | F1 | Останній training loss | Validation loss |
| ---: | ---: | ---: | ---: | ---: | ---: |
| 3 | 85,64 | 0,8603 | 0,9016 | 0,1626 | 0,4165 |
| 5 | 109,85 | 0,8578 | 0,8986 | 0,0489 | 0,6116 |

Після 5 епох training loss зменшився, а validation loss зріс. Accuracy і F1 також трохи знизилися. Це ознака перенавчання. За результатами валідації обрав би запуск на 3 епохи.

In [ ]:
%%time
del trainer, model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

set_seed(42)
model_5 = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
trainer_5 = make_trainer(model_5, 5)
started = time.perf_counter()
trainer_5.train()
training_5_s = time.perf_counter() - started
history_5 = list(trainer_5.state.log_history)
evaluation_5 = trainer_5.evaluate()
comparison = pd.DataFrame([
    {"Епохи": 3, "Навчання, с": training_3_s, "Accuracy": evaluation_3["eval_accuracy"],
        "F1": evaluation_3["eval_f1"], "Validation loss": evaluation_3["eval_loss"]},
    {"Епохи": 5, "Навчання, с": training_5_s, "Accuracy": evaluation_5["eval_accuracy"],
        "F1": evaluation_5["eval_f1"], "Validation loss": evaluation_5["eval_loss"]},
])
display(comparison.round(4))
for epochs, history in [(3, history_3), (5, history_5)]:
    print(f"Запуск на {epochs} епохах")
    display(pd.DataFrame([row for row in history if "loss" in row or "eval_loss" in row]).round(4))

bert_measurements = {"device": device, "train": len(tokenized_datasets["train"]),
    "validation": len(tokenized_datasets["validation"]), "comparison": comparison.to_dict("records"),
    "history_3": history_3, "history_5": history_5, "extra_pairs": extra_results,
    "mask_man": [(item["token_str"], item["score"]) for item in mask_man],
    "mask_woman": [(item["token_str"], item["score"]) for item in mask_woman]}
Path("bert_demo_measurements.json").write_text(json.dumps(bert_measurements, ensure_ascii=False, indent=2), encoding="utf-8")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'loss': '0.531', 'grad_norm': '12.84', 'learning_rate': '1.602e-05', 'epoch': '1'}
{'eval_loss': '0.3944', 'eval_accuracy': '0.8235', 'eval_f1': '0.8788', 'eval_runtime': '0.4543', 'eval_samples_per_second': '898.1', 'eval_steps_per_second': '15.41', 'epoch': '1'}
{'loss': '0.315', 'grad_norm': '9.09', 'learning_rate': '1.202e-05', 'epoch': '2'}
{'eval_loss': '0.3512', 'eval_accuracy': '0.8627', 'eval_f1': '0.9028', 'eval_runtime': '0.4712', 'eval_samples_per_second': '865.9', 'eval_steps_per_second': '14.86', 'epoch': '2'}
{'loss': '0.1718', 'grad_norm': '0.363', 'learning_rate': '8.017e-06', 'epoch': '3'}
{'eval_loss': '0.4753', 'eval_accuracy': '0.8676', 'eval_f1': '0.9062', 'eval_runtime': '0.937', 'eval_samples_per_second': '435.5', 'eval_steps_per_second': '7.471', 'epoch': '3'}
{'loss': '0.0914', 'grad_norm': '0.1676', 'learning_rate': '4.017e-06', 'epoch': '4'}
{'eval_loss': '0.6234', 'eval_accuracy': '0.848', 'eval_f1': '0.8938', 'eval_runtime': '0.4827', 'eval_samples_per_se

,Епохи,"Навчання, с",Accuracy,F1,Validation loss
0,3,85.6354,0.8603,0.9016,0.4165
1,5,109.8464,0.8578,0.8986,0.6116


Запуск на 3 епохах


,loss,grad_norm,learning_rate,epoch,step,eval_loss,eval_accuracy,eval_f1,eval_runtime,eval_samples_per_second,eval_steps_per_second
0,0.5317,15.3796,0.0,1.0,230,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,1.0,230,0.3815,0.8382,0.8870,0.4590,888.839,15.250
2,0.3101,4.8097,0.0,2.0,460,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,2.0,460,0.3576,0.8480,0.8935,0.4207,969.697,16.637
4,0.1626,0.9043,0.0,3.0,690,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,3.0,690,0.4165,0.8603,0.9016,0.4620,883.163,15.152


Запуск на 5 епохах


,loss,grad_norm,learning_rate,epoch,step,eval_loss,eval_accuracy,eval_f1,eval_runtime,eval_samples_per_second,eval_steps_per_second
0,0.5310,12.8423,0.0,1.0,230,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,1.0,230,0.3944,0.8235,0.8788,0.4543,898.132,15.409
2,0.3150,9.0900,0.0,2.0,460,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,2.0,460,0.3512,0.8627,0.9028,0.4712,865.936,14.857
4,0.1718,0.3630,0.0,3.0,690,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,3.0,690,0.4753,0.8676,0.9062,0.9370,435.455,7.471
6,0.0914,0.1676,0.0,4.0,920,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,NaN,NaN,4.0,920,0.6234,0.8480,0.8938,0.4827,845.220,14.501
8,0.0489,0.1029,0.0,5.0,1150,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,NaN,NaN,5.0,1150,0.6116,0.8578,0.8986,0.4680,871.732,14.956


CPU times: user 1min 37s, sys: 447 ms, total: 1min 37s
Wall time: 1min 51s


6081

### 10. Чотири власні перевірки

| Пара речень | Еталон | Прогноз | Імовірність парафразу |
| --- | ---: | ---: | ---: |
| Перефразований графік роботи бібліотеки | 1 | 1 | 0,8590 |
| Перефразоване повідомлення про перенесення засідання | 1 | 1 | 0,9777 |
| Кількість вакцинованих змінено зі 120 на 210 | 0 | 1 | 0,9846 |
| Журналіст і поліцейський помінялися ролями | 0 | 1 | 0,9878 |

Модель правильно класифікувала 2 з 4 пар. Вона помилково визнала парафразами обидві пари, у яких зміна числа або ролей учасників змінила зміст.